In [28]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.ml.feature import Imputer

# setup the configuration


In [29]:
# Local run: no MinIO/S3 needed since we're reading/writing local parquet files.
# (Keep the s3a config block commented out for when you move this back to the cluster.)
spark = (
    SparkSession.builder.appName("clean_dataset")
    .master("local[*]")
    # .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    # .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    # .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    # .config("spark.hadoop.fs.s3a.path.style.access", "true")
    # .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [30]:
IMPUTER_INPUT_COLS = ["anciennete_digitale_jours", "recence_gab_jours"]
IMPUTER_OUTPUT_COLS = ["anciennete_digitale_jours_imp", "recence_gab_jours_imp"]

# Local run: save the fitted Imputer model to a local folder instead of s3a://...
# (this was the cause of the crash: hadoop-aws jar isn't installed, so s3a:// paths fail)
IMPUTER_MODEL_PATH = "./models/imputer_anciennete_recence"


In [31]:
df = spark.read.parquet("part-00000.parquet")
df.show(100, truncate=False)


+-------+------+------+-------+------+----+-------------+----------+-------------------+----+--------+--------------+-------------+---------------+---------------+----------+-----------------+-----------+---------+-----------------------+-----------------------+-------------------+---------+----------+----------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+----------------------+-----------+----------------------+---------------------+--------------------+-------------------+----------------------+
|RADICAL|BANQUE|AGENCE|GENERIC|PLURAL|CCLE|DATE_OF_BIRTH|CODE_VILLE|LIBELLE_VILLE      |BPR |GENDER  |MARITAL_STATUS|NOMBRE_ENFANT|CUSTOMER_RATING|TAILLE_ENTREPRI|label_code|label_nom        |pack_actuel|pack_etat|digital_date_activation|digital_toujours_abonne|solde_moyen        |solde_min|solde_max |nb_mois_observes_solde|depot_moyen       |flux_cred_moyen   |flux_cred_total   |nb_mois_avec_f

# clean Data

In [32]:
df = df.withColumn(
    "GENDER",
    F.when(F.col("GENDER") == "FÃ©minin", "Féminin")
     .when(F.col("GENDER") == "Masculin", "Masculin")
     .otherwise(None)
)

In [33]:
for i, col in enumerate(df.columns):
    print(f"col {i}: {col} =>", df.filter(F.col(col).isNull()).count())


col 0: RADICAL => 0
col 1: BANQUE => 0
col 2: AGENCE => 0
col 3: GENERIC => 0
col 4: PLURAL => 0
col 5: CCLE => 0
col 6: DATE_OF_BIRTH => 0
col 7: CODE_VILLE => 0
col 8: LIBELLE_VILLE => 155
col 9: BPR => 2
col 10: GENDER => 1
col 11: MARITAL_STATUS => 0
col 12: NOMBRE_ENFANT => 267
col 13: CUSTOMER_RATING => 0
col 14: TAILLE_ENTREPRI => 4140
col 15: label_code => 0
col 16: label_nom => 0
col 17: pack_actuel => 987
col 18: pack_etat => 987
col 19: digital_date_activation => 1385
col 20: digital_toujours_abonne => 0
col 21: solde_moyen => 0
col 22: solde_min => 0
col 23: solde_max => 0
col 24: nb_mois_observes_solde => 0
col 25: depot_moyen => 605
col 26: flux_cred_moyen => 0
col 27: flux_cred_total => 0
col 28: nb_mois_avec_flux => 0
col 29: nb_operations_gab => 0
col 30: montant_total_gab => 0
col 31: montant_moyen_gab => 1950
col 32: derniere_operation_gab => 1950
col 33: nb_retraits => 0
col 34: montant_total_retraits => 0
col 35: nb_paiements_digitaux => 0
col 36: montant_total_pay

In [34]:
df.select("LIBELLE_VILLE").distinct().show(truncate=False)

+-----------------+
|LIBELLE_VILLE    |
+-----------------+
|AIT BAHA         |
|IMZOUREN         |
|AHFIR            |
|FARKHANA         |
|BIOUGRA          |
|ZEGANGANE        |
|ET-TAOUS (TAOUZ) |
|MHAYA            |
|CHTOUKA          |
|SEBT GZOULA      |
|NADOR            |
|BOUARFA          |
|SIDI SMAIL       |
|IFRANE           |
|ASSA             |
|ERFOUD           |
|TIFLET           |
|ERRACHIDIA       |
|AL HOCEIMA       |
|AFOURAR (AFOURER)|
+-----------------+
only showing top 20 rows


In [35]:
df.groupBy("LIBELLE_VILLE").count().show()

+-----------------+-----+
|    LIBELLE_VILLE|count|
+-----------------+-----+
|         AIT BAHA|    5|
|         IMZOUREN|    2|
|            AHFIR|    6|
|         FARKHANA|    1|
|          BIOUGRA|    9|
|        ZEGANGANE|    6|
| ET-TAOUS (TAOUZ)|    2|
|            MHAYA|    2|
|          CHTOUKA|    2|
|      SEBT GZOULA|    1|
|            NADOR|   49|
|          BOUARFA|    8|
|       SIDI SMAIL|    1|
|           IFRANE|    6|
|             ASSA|    6|
|           ERFOUD|    7|
|           TIFLET|   10|
|       ERRACHIDIA|   22|
|       AL HOCEIMA|   13|
|AFOURAR (AFOURER)|    2|
+-----------------+-----+
only showing top 20 rows


# Script

In [36]:
def clean_dataset(df: DataFrame) -> DataFrame:
    """
    Applique toutes les règles de nettoyage décidées pour ce projet.
    Chaque étape est commentée avec la décision métier qui la justifie
    (voir section 6.5 / 6.5bis du guide maître pour le détail).
    """

    n_avant = df.count()

    # --- 1. LIBELLE_VILLE : redondant avec CODE_VILLE (le code est fiable,
    #        0 null) -> on garde le code, on jette le libellé plutôt que
    #        d'imputer un texte qui n'apporte rien de plus au modèle ---
    if "LIBELLE_VILLE" in df.columns:
        df = df.drop("LIBELLE_VILLE")

    # --- 2. BPR / GENDER : nulls négligeables (2 et 1 lignes sur l'échantillon
    #        de référence) -> on supprime juste ces lignes, pas d'imputation ---
    subset_dropna = [c for c in ["BPR", "GENDER"] if c in df.columns]
    if subset_dropna:
        df = df.dropna(subset=subset_dropna)

    # --- 3. NOMBRE_ENFANT : null = pas d'enfant, PAS une valeur manquante
    #        à imputer par une médiane -> fillna(0) directement ---
    if "NOMBRE_ENFANT" in df.columns:
        df = df.fillna({"NOMBRE_ENFANT": 0})

    # --- 4. TAILLE_ENTREPRI : null = compte particulier (pas d'entreprise),
    #        pas une donnée manquante -> valeur catégorielle explicite.
    #        Devient un signal utile (particulier vs professionnel) au lieu
    #        d'être dropé ou imputé avec du bruit ---
    if "TAILLE_ENTREPRI" in df.columns:
        df = df.fillna({"TAILLE_ENTREPRI": "PARTICULIER"})

    # --- 4bis. pack_actuel / pack_etat : nulls groupés sur les mêmes clients
    #           (986 chacun) = clients sans pack digital, pas une vraie
    #           valeur manquante -> catégorie explicite, pas le mode ---
    pack_cols = {}
    if "pack_actuel" in df.columns:
        pack_cols["pack_actuel"] = "SANS_PACK"
    if "pack_etat" in df.columns:
        pack_cols["pack_etat"] = "SANS_ETAT"
    if pack_cols:
        df = df.fillna(pack_cols)

    # --- 5. depot_moyen / montant_moyen_gab : null = absence d'activité
    #        observée -> 0, cohérent avec le traitement déjà appliqué à
    #        flux_cred_moyen (NaN = 0, jamais une moyenne) ---
    montants_zero = [c for c in ["depot_moyen", "montant_moyen_gab"] if c in df.columns]
    if montants_zero:
        df = df.fillna({c: 0.0 for c in montants_zero})

    # --- 6. digital_date_activation : une date ne peut pas être mise à 0
    #        (serait confondu avec "activé aujourd'hui"). On dérive une
    #        ancienneté en jours + un flag explicite, puis on jette la
    #        date brute qui n'est de toute façon pas exploitable telle
    #        quelle par VectorAssembler ---
    if "digital_date_activation" in df.columns:
        df = (
            df.withColumn(
                "jamais_active_digital",
                F.when(F.col("digital_date_activation").isNull(), 1).otherwise(0),
            )
            .withColumn(
                "anciennete_digitale_jours",
                F.when(F.col("digital_date_activation").isNull(), F.lit(None))
                .otherwise(
                    F.datediff(
                        F.current_date(),
                        F.to_date("digital_date_activation", "dd/MM/yyyy"),
                    )
                ),
            )
            .drop("digital_date_activation")
        )

    # --- 7. derniere_operation_gab : même logique que #6, format datetime ---
    if "derniere_operation_gab" in df.columns:
        df = (
            df.withColumn(
                "jamais_utilise_gab",
                F.when(F.col("derniere_operation_gab").isNull(), 1).otherwise(0),
            )
            .withColumn(
                "recence_gab_jours",
                F.when(F.col("derniere_operation_gab").isNull(), F.lit(None))
                .otherwise(
                    F.datediff(
                        F.current_date(),
                        F.to_date(
                            F.col("derniere_operation_gab"), "dd/MM/yyyy HH:mm:ss"
                        ),
                    )
                ),
            )
            .drop("derniere_operation_gab")
        )

    n_apres = df.count()
    print(f"    Lignes avant : {n_avant} | après (dropna BPR/GENDER) : {n_apres}")

    return df

In [37]:
def run(path_in: str, path_out: str, label: str):
    print(f"\n{'=' * 20} NETTOYAGE : {label} {'=' * 20}")
    print(f"Lecture : {path_in}")
    df = spark.read.parquet(path_in)

    print("Nulls avant nettoyage :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    df_clean = clean_dataset(df)

    print("Nulls après nettoyage :")
    df_clean.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]
    ).show(truncate=False, vertical=True)

    print(f"Écriture : {path_out}")
    df_clean.write.mode("overwrite").parquet(path_out)
    print(f"OK : {label} nettoyé et écrit.\n")


In [38]:
def apply_base_cleaning(path_in: str, label: str) -> DataFrame:
    print(f"\n{'=' * 20} NETTOYAGE DE BASE : {label} {'=' * 20}")
    print(f"Lecture : {path_in}")
    df = spark.read.parquet(path_in)

    print("Nulls avant nettoyage :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    df_clean = clean_dataset(df)
    return df_clean

In [39]:
def fit_and_apply_imputer_on_train(df_train: DataFrame) -> DataFrame:
    """
    Option A : Imputer (médiane) sur anciennete_digitale_jours / recence_gab_jours.
    Fit UNIQUEMENT sur le train (clients_avec_label) -- jamais sur la population
    à scorer, pour éviter toute fuite d'information et rester cohérent avec la
    logique déjà utilisée pour le Pipeline MLlib (fit sur train, transform partout).
    Le modèle d'imputation est sauvegardé pour être réappliqué tel quel sur
    dataset_a_scorer (mêmes médianes, pas recalculées).

    Les colonnes brutes (anciennete_digitale_jours / recence_gab_jours) sont
    droppées après imputation : seules les versions *_imp (sans null) doivent
    entrer dans le VectorAssembler, aux côtés des flags jamais_active_digital /
    jamais_utilise_gab qui permettent au modèle de distinguer une vraie valeur
    d'une médiane fictive.
    """
    cols_present = [c for c in IMPUTER_INPUT_COLS if c in df_train.columns]
    if not cols_present:
        return df_train

    out_cols = [IMPUTER_OUTPUT_COLS[IMPUTER_INPUT_COLS.index(c)] for c in cols_present]

    imputer = Imputer(inputCols=cols_present, outputCols=out_cols, strategy="median")
    imputer_model = imputer.fit(df_train)

    medianes = {c: df_train.approxQuantile(c, [0.5], 0.01)[0] for c in cols_present}
    print(f"Médianes apprises sur le train : {medianes}")

    df_train_imp = imputer_model.transform(df_train)

    print(f"Sauvegarde du modèle d'imputation : {IMPUTER_MODEL_PATH}")
    imputer_model.write().overwrite().save(IMPUTER_MODEL_PATH)

    # Drop des colonnes brutes une fois les versions _imp calculées
    df_train_imp = df_train_imp.drop(*cols_present)

    return df_train_imp


In [40]:
def apply_saved_imputer(df_scorer: DataFrame) -> DataFrame:
    """Recharge l'Imputer entraîné sur le train et l'applique tel quel au
    dataset à scorer -- mêmes médianes des deux côtés, aucune fuite.
    Droppe aussi les colonnes brutes après imputation, comme sur le train."""
    from pyspark.ml.feature import ImputerModel

    cols_present = [c for c in IMPUTER_INPUT_COLS if c in df_scorer.columns]
    if not cols_present:
        return df_scorer

    print(f"Chargement du modèle d'imputation : {IMPUTER_MODEL_PATH}")
    imputer_model = ImputerModel.load(IMPUTER_MODEL_PATH)
    df_scorer_imp = imputer_model.transform(df_scorer)

    # Drop des colonnes brutes une fois les versions _imp calculées
    df_scorer_imp = df_scorer_imp.drop(*cols_present)

    return df_scorer_imp


In [41]:
def show_nulls_and_write(df: DataFrame, path_out: str, label: str):
    print("Nulls après nettoyage complet (base + imputation) :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    print(f"Écriture : {path_out}")
    df.write.mode("overwrite").parquet(path_out)
    print(f"OK : {label} nettoyé et écrit.\n")

In [42]:
if __name__ == "__main__":
    import os

    # Adapter ce chemin si votre nom de dossier Parquet diffère
    PATH_TRAIN_IN = "part-00000.parquet"

    # Nom de sortie = nom du fichier d'entrée + "_clean"
    # ex: "part-00000.parquet" -> "part-00000_clean.parquet"
    _base, _ext = os.path.splitext(PATH_TRAIN_IN)
    PATH_TRAIN_OUT = f"{_base}_clean{_ext}"

    # 1. Nettoyage de base + fit de l'Imputer SUR LE TRAIN UNIQUEMENT
    df_train = apply_base_cleaning(PATH_TRAIN_IN, "dataset_train_produits (clients avec label)")
    df_train = fit_and_apply_imputer_on_train(df_train)
    show_nulls_and_write(df_train, PATH_TRAIN_OUT, "dataset_train_produits (clients avec label)")

    print("\nTerminé. anciennete_digitale_jours_imp / recence_gab_jours_imp sont")
    print("désormais sans null (médianes apprises sur le train, réutilisées pour")
    print("le scoring). Utiliser les colonnes *_imp (pas les brutes) dans le")
    print("VectorAssembler, aux côtés des flags jamais_active_digital / jamais_utilise_gab.")
    print(f"\nFichier de sortie : {PATH_TRAIN_OUT}")



==================== NETTOYAGE DE BASE : dataset_train_produits (clients avec label) ====================
Lecture : part-00000.parquet
Nulls avant nettoyage :
-RECORD 0-----------------------
 RADICAL                 | 0    
 BANQUE                  | 0    
 AGENCE                  | 0    
 GENERIC                 | 0    
 PLURAL                  | 0    
 CCLE                    | 0    
 DATE_OF_BIRTH           | 0    
 CODE_VILLE              | 0    
 LIBELLE_VILLE           | 155  
 BPR                     | 2    
 GENDER                  | 1    
 MARITAL_STATUS          | 0    
 NOMBRE_ENFANT           | 267  
 CUSTOMER_RATING         | 0    
 TAILLE_ENTREPRI         | 4140 
 label_code              | 0    
 label_nom               | 0    
 pack_actuel             | 987  
 pack_etat               | 987  
 digital_date_activation | 1385 
 digital_toujours_abonne | 0    
 solde_moyen             | 0    
 solde_min               | 0    
 solde_max               | 0    
 nb_mois_observ